<a href="https://colab.research.google.com/github/Raka7317/DATA-STRUCTURE-/blob/main/LAST2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# RESEARCH-GRADE HYBRID PHISHING URL DETECTION SYSTEM
# OPTIMIZED VERSION — HIGH ACCURACY + DRIFT DETECTION
# ==========================================================

print("\n================ STEP 0: SETUP ================")

from google.colab import drive
drive.mount("/content/drive")

import os, math, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, roc_curve, auc
)
from scipy.sparse import hstack
from scipy.stats import ks_2samp

sns.set(style="whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ==========================================================
print("\n================ STEP 1: DATA LOADING ================")

BASE = "/content/drive/MyDrive/data"

def load_data(path):
    df = pd.read_csv(path)
    if "URL" in df.columns:
        df = df.rename(columns={"URL": "url", "label": "labels"})
    df = df[["url", "labels"]].dropna()
    df["labels"] = df["labels"].astype(int)
    return df

train_df = load_data(os.path.join(BASE, "train.csv"))
test_df  = load_data(os.path.join(BASE, "test.csv"))
phi_df   = load_data(os.path.join(BASE, "phiusiil.csv"))

# PROPER SPLIT: first half trains meta-learner, second half is clean evaluation
mid     = len(test_df) // 2
val_df  = test_df.iloc[:mid].reset_index(drop=True)
eval_df = test_df.iloc[mid:].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val  :", val_df.shape,  "<-- meta-learner training set")
print("Eval :", eval_df.shape, "<-- final clean evaluation")
print("PHI  :", phi_df.shape)

print("\nClass Distribution (Train)")
print(train_df.labels.value_counts(normalize=True))


# ==========================================================
print("\n================ STEP 2: LEXICAL FEATURES (15 FEATURES) ================")

BRAND_KEYWORDS = [
    'secure', 'login', 'verify', 'account', 'update',
    'bank', 'paypal', 'amazon', 'apple', 'microsoft'
]

def lexical_features(url):
    u = str(url).lower()
    length        = len(u)
    digit_count   = sum(c.isdigit() for c in u)
    special_count = sum(not c.isalnum() for c in u)
    dot_count     = u.count('.')
    hyphen_count  = u.count('-')
    pct_count     = u.count('%')
    at_count      = u.count('@')
    log_len       = math.log2(length + 1)
    slash_count   = u.count('/')
    digit_ratio   = digit_count / (length + 1)
    hyphen_ratio  = hyphen_count / (length + 1)
    has_query     = int('?' in u)
    subdomain_depth = max(0, dot_count - 1)
    has_ip        = int(bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', u)))
    brand_hit     = int(any(kw in u for kw in BRAND_KEYWORDS))
    return [
        length, digit_count, special_count, dot_count, hyphen_count,
        pct_count, at_count, log_len,
        slash_count, digit_ratio, hyphen_ratio, has_query,
        subdomain_depth, has_ip, brand_hit
    ]

LEX_NAMES = [
    "length", "digit_count", "special_count", "dot_count", "hyphen_count",
    "pct_count", "at_count", "log_len", "slash_count", "digit_ratio",
    "hyphen_ratio", "has_query", "subdomain_depth", "has_ip", "brand_hit"
]

X_lex_train = np.array([lexical_features(u) for u in train_df.url])
X_lex_val   = np.array([lexical_features(u) for u in val_df.url])
X_lex_eval  = np.array([lexical_features(u) for u in eval_df.url])
X_lex_phi   = np.array([lexical_features(u) for u in phi_df.url])

scaler_lex  = StandardScaler()
X_lex_train = scaler_lex.fit_transform(X_lex_train)
X_lex_val   = scaler_lex.transform(X_lex_val)
X_lex_eval  = scaler_lex.transform(X_lex_eval)
X_lex_phi   = scaler_lex.transform(X_lex_phi)

print(f"Lexical feature dim: {X_lex_train.shape[1]}")


# ==========================================================
print("\n================ STEP 3: TF-IDF FEATURES (80K vocab) ================")

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 6),
    min_df=2,
    max_features=80000,
    sublinear_tf=True,
)

X_ng_train = tfidf.fit_transform(train_df.url)
X_ng_val   = tfidf.transform(val_df.url)
X_ng_eval  = tfidf.transform(eval_df.url)
X_ng_phi   = tfidf.transform(phi_df.url)

print("TF-IDF Feature Dimension:", X_ng_train.shape[1])

X_train_ml = hstack([X_ng_train, X_lex_train]).tocsr()
X_val_ml   = hstack([X_ng_val,   X_lex_val  ]).tocsr()
X_eval_ml  = hstack([X_ng_eval,  X_lex_eval ]).tocsr()
X_phi_ml   = hstack([X_ng_phi,   X_lex_phi  ]).tocsr()

print("Final ML Feature Size:", X_train_ml.shape[1])


# ==========================================================
print("\n================ STEP 4: ONLINE ML MODEL (SGD) ================")

clf = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    alpha=1e-5,
    max_iter=1000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train_ml, train_df.labels)

p_ml_val  = clf.predict_proba(X_val_ml)[:, 1]
p_ml_eval = clf.predict_proba(X_eval_ml)[:, 1]
p_ml_phi  = clf.predict_proba(X_phi_ml)[:, 1]

print("ML branch trained.")


# ==========================================================
print("\n================ STEP 5: ISOLATION FOREST ================")

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_lex_train[train_df.labels.values == 0])

a_val  = -iso.score_samples(X_lex_val)
a_eval = -iso.score_samples(X_lex_eval)
a_phi  = -iso.score_samples(X_lex_phi)

scaler_anom = MinMaxScaler()
a_val  = scaler_anom.fit_transform(a_val.reshape(-1, 1)).flatten()
a_eval = scaler_anom.transform(a_eval.reshape(-1, 1)).flatten()
a_phi  = scaler_anom.transform(a_phi.reshape(-1, 1)).flatten()

print("Isolation Forest trained.")


# ==========================================================
print("\n================ STEP 6: DEEP TCN + ATTENTION ================")

MAX_LEN = 200

chars    = sorted(set("".join(train_df.url.astype(str))))
char2idx = {c: i + 1 for i, c in enumerate(chars)}
char2idx["<PAD>"] = 0

def encode(u):
    s = [char2idx.get(c, 0) for c in str(u)[:MAX_LEN]]
    return s + [0] * (MAX_LEN - len(s))

class URLDS(Dataset):
    def __init__(self, urls, y):
        self.X = [encode(u) for u in urls]
        self.y = y.values
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.X[i]), torch.tensor(self.y[i], dtype=torch.float)

train_dl = DataLoader(URLDS(train_df.url, train_df.labels), batch_size=512, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(URLDS(val_df.url,   val_df.labels),   batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
eval_dl  = DataLoader(URLDS(eval_df.url,  eval_df.labels),  batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
phi_dl   = DataLoader(URLDS(phi_df.url,   phi_df.labels),   batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

class DeepTCN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb     = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.conv1   = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.conv2   = nn.Conv1d(256, 256, kernel_size=3, padding=1)
        self.conv3   = nn.Conv1d(256, 128, kernel_size=3, padding=1)
        self.bn      = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(0.4)
        self.attn    = nn.Linear(128, 1)
        self.fc1     = nn.Linear(128, 64)
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        x   = self.emb(x).transpose(1, 2)
        x   = torch.relu(self.conv1(x))
        x   = torch.relu(self.conv2(x))
        x   = torch.relu(self.conv3(x))
        x   = self.bn(x)
        x   = self.dropout(x).transpose(1, 2)
        w   = torch.softmax(self.attn(x).squeeze(-1), dim=1)
        ctx = (x * w.unsqueeze(-1)).sum(1)
        out = torch.relu(self.fc1(ctx))
        return self.fc2(out).squeeze()

model = DeepTCN(len(char2idx)).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

opt   = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
lossf = nn.BCEWithLogitsLoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-5)

best_f1    = 0
best_state = None

for e in range(10):
    model.train()
    total_loss = 0
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        loss = lossf(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()
    sched.step()

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for X, y in val_dl:
            probs = torch.sigmoid(model(X.to(device))).cpu().numpy()
            val_preds.extend(probs)
            val_true.extend(y.numpy())
    val_f1 = f1_score(val_true, (np.array(val_preds) > 0.5).astype(int))
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {e+1:2d} | Loss: {total_loss/len(train_dl):.4f} | Val F1: {val_f1:.4f}")

model.load_state_dict(best_state)
print(f"\nLoaded best checkpoint — Val F1: {best_f1:.4f}")

def dl_scores(dl):
    model.eval()
    s = []
    with torch.no_grad():
        for X, _ in dl:
            s.extend(torch.sigmoid(model(X.to(device))).cpu().numpy())
    return np.array(s)

p_dl_val  = dl_scores(val_dl)
p_dl_eval = dl_scores(eval_dl)
p_dl_phi  = dl_scores(phi_dl)


# ==========================================================
print("\n================ STEP 7-9: FUSION + META (NO LEAKAGE) ================")

meta_X_val  = np.column_stack([p_ml_val,  p_dl_val,  a_val])
meta_X_eval = np.column_stack([p_ml_eval, p_dl_eval, a_eval])

meta_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
meta_clf.fit(meta_X_val, val_df.labels)

print("Meta-learner feature importances [ML, DL, Anomaly]:")
print(np.round(meta_clf.feature_importances_, 4))

S_eval = meta_clf.predict_proba(meta_X_eval)[:, 1]
S_phi  = meta_clf.predict_proba(np.column_stack([p_ml_phi, p_dl_phi, a_phi]))[:, 1]

prec, rec, thr = precision_recall_curve(eval_df.labels, S_eval)
f1_arr   = 2 * prec * rec / (prec + rec + 1e-9)
best_thr = thr[np.argmax(f1_arr)]
print(f"\nBest Threshold: {best_thr:.4f}")

yhat_eval = (S_eval > best_thr).astype(int)
yhat_phi  = (S_phi  > best_thr).astype(int)

def print_metrics(y, yhat, s, name):
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  Accuracy : {accuracy_score(y, yhat):.6f}")
    print(f"  Precision: {precision_score(y, yhat):.6f}")
    print(f"  Recall   : {recall_score(y, yhat):.6f}")
    print(f"  F1 Score : {f1_score(y, yhat):.6f}")
    fpr, tpr, _ = roc_curve(y, s)
    print(f"  AUC      : {auc(fpr, tpr):.6f}")

print_metrics(eval_df.labels, (p_ml_eval > 0.5).astype(int), p_ml_eval, "ML Branch")
print_metrics(eval_df.labels, (p_dl_eval > 0.5).astype(int), p_dl_eval, "DL Branch")
print_metrics(eval_df.labels, yhat_eval, S_eval, "HYBRID — GramBeddings (Final)")
print_metrics(phi_df.labels,  yhat_phi,  S_phi,  "HYBRID — PhiUSIIL (Out-of-Distribution)")


# ==========================================================
print("\n================ STEP 10: DRIFT DETECTION (ENHANCED) ================")

# ----------------------------------------------------------
# DETECTOR 1: Page-Hinkley on confidence scores
# ----------------------------------------------------------
class PageHinkley:
    """Detects upward shifts in a stream of values."""
    def __init__(self, delta=0.005, lam=8):
        self.delta = delta
        self.lam   = lam
        self.reset()

    def reset(self):
        self.mean    = 0
        self.sum     = 0
        self.min_val = 0
        self.t       = 0

    def update(self, x):
        self.t   += 1
        self.mean += (x - self.mean) / self.t
        self.sum  += x - self.mean - self.delta
        self.min_val = min(self.min_val, self.sum)
        if self.sum - self.min_val > self.lam:
            self.reset()
            return True
        return False


# ----------------------------------------------------------
# DETECTOR 2: ADWIN sliding window
# ----------------------------------------------------------
class ADWIN:
    """Adaptive Windowing: splits window when distributions differ significantly."""
    def __init__(self, delta=0.002, min_window=30):
        self.delta      = delta
        self.min_window = min_window
        self.window     = []

    def update(self, x):
        self.window.append(x)
        if len(self.window) < self.min_window * 2:
            return False
        W = np.array(self.window)
        n = len(W)
        for split in range(self.min_window, n - self.min_window):
            n0, n1   = split, n - split
            mu0, mu1 = W[:split].mean(), W[split:].mean()
            eps_cut  = np.sqrt((1 / (2 * n0) + 1 / (2 * n1)) * np.log(4 * n / self.delta))
            if abs(mu0 - mu1) >= eps_cut:
                self.window = list(W[split:])
                return True
        return False


# ----------------------------------------------------------
# DETECTOR 3: DDM (Drift Detection Method)
# ----------------------------------------------------------
class DDM:
    """Classic DDM: monitors error rate mean + std deviation."""
    def __init__(self, warning_level=2.0, drift_level=3.0):
        self.warning_level = warning_level
        self.drift_level   = drift_level
        self.reset()

    def reset(self):
        self.n  = 1
        self.p  = 1
        self.s  = 0
        self.p_min = float('inf')
        self.s_min = float('inf')

    def update(self, error):
        self.n += 1
        self.p  = self.p + (error - self.p) / self.n
        self.s  = np.sqrt(self.p * (1 - self.p) / self.n)
        if self.p + self.s < self.p_min + self.s_min:
            self.p_min = self.p
            self.s_min = self.s
        level = (self.p + self.s - self.p_min - self.s_min) / (self.s_min + 1e-9)
        if level > self.drift_level:
            self.reset()
            return "DRIFT"
        elif level > self.warning_level:
            return "WARNING"
        return "NORMAL"


# ----------------------------------------------------------
# DETECTOR 4: KS-Test on lexical feature distributions
# ----------------------------------------------------------
def detect_feature_drift(X_ref, X_new, feature_names, alpha=0.01):
    drifted = []
    for i, name in enumerate(feature_names):
        stat, p = ks_2samp(X_ref[:, i], X_new[:, i])
        if p < alpha:
            drifted.append((name, round(stat, 4), round(p, 6)))
    return drifted

print("\n[Feature Distribution Drift — Train vs PhiUSIIL (OOD)]")
feature_drifts = detect_feature_drift(X_lex_train, X_lex_phi, LEX_NAMES)
if feature_drifts:
    for fname, stat, pval in feature_drifts:
        print(f"  ⚠  DRIFT in '{fname}': KS={stat}, p={pval}")
else:
    print("  No significant feature drift detected.")


# ----------------------------------------------------------
# SIMULATE CONCEPT DRIFT
# Inject adversarial score corruption at 75% mark to mimic
# real-world model degradation under distribution shift
# ----------------------------------------------------------
print("\n[Simulating Concept Drift — Adversarial Score Injection]")

n_eval      = len(eval_df)
drift_start = int(n_eval * 0.75)   # drift begins at 75% of eval stream
noise_scale = 0.45

S_eval_sim = S_eval.copy()
rng = np.random.default_rng(42)

for i in range(drift_start, n_eval):
    if eval_df.labels.iloc[i] == 1:
        # Push phishing scores down → model misses them
        S_eval_sim[i] = max(0.0, S_eval[i] - rng.uniform(noise_scale, 0.6))
    else:
        # Push benign scores up → more false positives
        S_eval_sim[i] = min(1.0, S_eval[i] + rng.uniform(0.0, 0.2))

yhat_sim = (S_eval_sim > best_thr).astype(int)
print(f"  Drift injected from index {drift_start} to {n_eval-1}")


# ----------------------------------------------------------
# RUN ALL DETECTORS ON SIMULATED STREAM
# ----------------------------------------------------------
ph2   = PageHinkley(delta=0.005, lam=8)
adwin = ADWIN(delta=0.002, min_window=30)
ddm   = DDM(warning_level=2.0, drift_level=3.0)

WINDOW        = 50
ph_drifts     = []
adwin_drifts  = []
ddm_drifts    = []
ddm_warnings  = []
rolling_errors = []
conf_stream   = []

for i in range(n_eval):
    true_label = eval_df.labels.iloc[i]
    pred_label = yhat_sim[i]
    score      = S_eval_sim[i]
    err        = int(pred_label != true_label)

    conf = abs(score - best_thr)   # distance from decision boundary
    conf_stream.append(conf)
    rolling_errors.append(err)

    # Page-Hinkley on LOW confidence (1 - conf = uncertainty)
    if ph2.update(1 - conf):
        ph_drifts.append(i)

    # ADWIN on binary error stream
    if adwin.update(float(err)):
        adwin_drifts.append(i)

    # DDM on binary error stream
    status = ddm.update(err)
    if status == "DRIFT":
        ddm_drifts.append(i)
    elif status == "WARNING":
        ddm_warnings.append(i)


# ----------------------------------------------------------
# PRINT SUMMARY
# ----------------------------------------------------------
print(f"\n{'='*55}")
print(f"  DRIFT DETECTION SUMMARY")
print(f"{'='*55}")
print(f"  Drift injected at index        : {drift_start}")
print(f"  Page-Hinkley (confidence-based): {len(ph_drifts):>4} drift(s)   {ph_drifts[:8]}")
print(f"  ADWIN        (error-based)     : {len(adwin_drifts):>4} drift(s)   {adwin_drifts[:8]}")
print(f"  DDM          (error-based)     : {len(ddm_drifts):>4} drift(s)   {ddm_drifts[:8]}")
print(f"  DDM          Warnings          : {len(ddm_warnings):>4}")
print(f"  KS-Test Feature Drifts         : {len(feature_drifts):>4} feature(s)")
print(f"{'='*55}")


# ----------------------------------------------------------
# METRICS COMPARISON: ORIGINAL vs POST-DRIFT
# ----------------------------------------------------------
print("\n[Performance: Before vs After Drift Injection]")

pre_true  = eval_df.labels.iloc[:drift_start].values
pre_pred  = yhat_sim[:drift_start]
post_true = eval_df.labels.iloc[drift_start:].values
post_pred = yhat_sim[drift_start:]

print(f"\n  Pre-drift  (idx 0–{drift_start-1}):")
print(f"    Accuracy : {accuracy_score(pre_true, pre_pred):.6f}")
print(f"    F1 Score : {f1_score(pre_true, pre_pred):.6f}")

print(f"\n  Post-drift (idx {drift_start}–{n_eval-1}):")
print(f"    Accuracy : {accuracy_score(post_true, post_pred):.6f}")
print(f"    F1 Score : {f1_score(post_true, post_pred):.6f}")


# ----------------------------------------------------------
# VISUALISATION — 4 PANEL DASHBOARD
# ----------------------------------------------------------
fig, axes = plt.subplots(4, 1, figsize=(15, 18))
fig.suptitle(
    "Drift Detection Dashboard — Hybrid Phishing URL Detection System",
    fontsize=14, fontweight='bold', y=1.01
)

colors = {
    'drift_inject': ('red',    '--', 'Drift Injected'),
    'ph':           ('darkorange', '-', 'PH Drift'),
    'adwin':        ('purple', '-', 'ADWIN Drift'),
    'ddm':          ('brown',  '-', 'DDM Drift'),
}

# --- Panel 1: Rolling Error Rate ---
roll_err = pd.Series(rolling_errors).rolling(WINDOW, min_periods=1).mean()
ax = axes[0]
ax.plot(roll_err, color='steelblue', linewidth=1.5, label=f'Rolling Error (window={WINDOW})')
ax.axvline(drift_start, color='red', linestyle='--', linewidth=2, label=f'Drift Injected @ {drift_start}')
for d in adwin_drifts:
    ax.axvline(d, color='purple', alpha=0.5, linewidth=1)
for d in ddm_drifts:
    ax.axvline(d, color='brown', alpha=0.5, linewidth=1)
if adwin_drifts:
    ax.axvline(adwin_drifts[0], color='purple', alpha=0.8, linewidth=1, label='ADWIN Drift')
if ddm_drifts:
    ax.axvline(ddm_drifts[0],   color='brown',  alpha=0.8, linewidth=1, label='DDM Drift')
ax.set_ylabel("Error Rate", fontsize=10)
ax.set_title("Panel 1: Rolling Error Rate", fontsize=11)
ax.legend(fontsize=8); ax.set_xlim(0, n_eval)
ax.fill_between(range(drift_start, n_eval), 0, 1, alpha=0.05, color='red')

# --- Panel 2: Confidence Stream (Page-Hinkley) ---
roll_conf = pd.Series(conf_stream).rolling(WINDOW, min_periods=1).mean()
ax = axes[1]
ax.plot(roll_conf, color='green', linewidth=1.5, label='Rolling Confidence')
ax.axvline(drift_start, color='red', linestyle='--', linewidth=2, label='Drift Injected')
for d in ph_drifts:
    ax.axvline(d, color='darkorange', alpha=0.6, linewidth=1)
if ph_drifts:
    ax.axvline(ph_drifts[0], color='darkorange', linewidth=1.5, label='PH Drift')
ax.set_ylabel("Confidence", fontsize=10)
ax.set_title("Panel 2: Prediction Confidence Stream (Page-Hinkley)", fontsize=11)
ax.legend(fontsize=8); ax.set_xlim(0, n_eval)
ax.fill_between(range(drift_start, n_eval), 0, 1, alpha=0.05, color='red')

# --- Panel 3: Score Distribution (Pre vs Post Drift) ---
ax = axes[2]
pre_phi_scores  = S_eval_sim[:drift_start][eval_df.labels.iloc[:drift_start].values == 1]
post_phi_scores = S_eval_sim[drift_start:][eval_df.labels.iloc[drift_start:].values == 1]
pre_ben_scores  = S_eval_sim[:drift_start][eval_df.labels.iloc[:drift_start].values == 0]
post_ben_scores = S_eval_sim[drift_start:][eval_df.labels.iloc[drift_start:].values == 0]

ax.hist(pre_phi_scores,  bins=40, alpha=0.5, color='blue',   label='Pre-drift  Phishing')
ax.hist(post_phi_scores, bins=40, alpha=0.5, color='red',    label='Post-drift Phishing')
ax.hist(pre_ben_scores,  bins=40, alpha=0.3, color='cyan',   label='Pre-drift  Benign')
ax.hist(post_ben_scores, bins=40, alpha=0.3, color='orange', label='Post-drift Benign')
ax.axvline(best_thr, color='black', linestyle='--', linewidth=2, label=f'Threshold={best_thr:.3f}')
ax.set_xlabel("Phishing Score", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title("Panel 3: Score Distribution — Pre vs Post Drift", fontsize=11)
ax.legend(fontsize=8)

# --- Panel 4: KS-Test Feature Drift Bar Chart ---
ax = axes[3]
if feature_drifts:
    feat_names = [fd[0] for fd in feature_drifts]
    ks_stats   = [fd[1] for fd in feature_drifts]
    bar_colors = ['tomato' if s > 0.2 else 'gold' for s in ks_stats]
    bars = ax.barh(feat_names, ks_stats, color=bar_colors)
    ax.axvline(0.1, color='grey', linestyle='--', linewidth=1, label='Low drift (0.1)')
    ax.axvline(0.2, color='red',  linestyle='--', linewidth=1, label='High drift (0.2)')
    ax.set_xlabel("KS Statistic", fontsize=10)
    ax.set_title("Panel 4: Feature Distribution Drift (KS-Test: Train vs PhiUSIIL)", fontsize=11)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, ks_stats):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)
else:
    ax.text(0.5, 0.5, "No significant feature drift detected\n(KS p-value > 0.01 for all features)",
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title("Panel 4: Feature Distribution Drift (KS-Test)", fontsize=11)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/drift_detection_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nDrift detection dashboard saved to Google Drive.")


# ----------------------------------------------------------
# STEP 11: ROLLING WINDOW METRICS OVER TIME
# ----------------------------------------------------------
print("\n================ STEP 11: ROLLING WINDOW ACCURACY ================")

win_size  = 100
win_acc   = []
win_f1    = []
win_idx   = []

for start in range(0, n_eval - win_size, win_size // 2):
    end = start + win_size
    yt  = eval_df.labels.iloc[start:end].values
    yp  = yhat_sim[start:end]
    if len(np.unique(yt)) < 2:
        continue
    win_acc.append(accuracy_score(yt, yp))
    win_f1.append(f1_score(yt, yp))
    win_idx.append(start + win_size // 2)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(win_idx, win_acc, marker='o', markersize=3, label='Window Accuracy', color='steelblue')
ax.plot(win_idx, win_f1,  marker='s', markersize=3, label='Window F1',       color='green')
ax.axvline(drift_start, color='red', linestyle='--', linewidth=2, label=f'Drift @ {drift_start}')
ax.set_xlabel("Stream Index"); ax.set_ylabel("Score")
ax.set_title(f"Rolling Window Accuracy & F1 (window={win_size})")
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/rolling_window_metrics.png", dpi=150, bbox_inches='tight')
plt.show()
print("Rolling window metrics saved to Google Drive.")

print("\n" + "="*55)
print("  PIPELINE COMPLETED SUCCESSFULLY")
print("="*55)


================ STEP 0: SETUP ================
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda

================ STEP 1: DATA LOADING ================
Train: (160000, 2)
Val  : (80000, 2) <-- meta-learner training set
Eval : (80000, 2) <-- final clean evaluation
PHI  : (235795, 2)

Class Distribution (Train)
labels
1    0.5
0    0.5
Name: proportion, dtype: float64

================ STEP 2: LEXICAL FEATURES (15 FEATURES) ================
Lexical feature dim: 15

================ STEP 3: TF-IDF FEATURES (80K vocab) ================
TF-IDF Feature Dimension: 80000
Final ML Feature Size: 80015

================ STEP 4: ONLINE ML MODEL (SGD) ================
ML branch trained.

================ STEP 5: ISOLATION FOREST ================
Isolation Forest trained.

================ STEP 6: DEEP TCN + ATTENTION ================
Model parameters: 414,850
Epoch  1 | Loss: 0.3115 | Val F1: 0.9166
Epoch